# 🧠 第 4 阶段：单步 Agent（One-shot Tool Use）

## 🎯 阶段目标

让模型**自动决定**是否调用工具（但只调用一次）。

> **单步 Agent = 思考 + 行动**

这是实现完整 Agent Loop 的重要一步。

---

## 🧠 核心概念

单步 Agent 流程：

```
用户输入 → LLM 判断 → 是否调用工具？
                        ↓
                    是 / 否
                    ↓
              执行工具 / 直接回答
                    ↓
              整理结果给用户
```

---

## 📦 1. 加载配置

In [1]:
import json
import requests
import re
from pathlib import Path

# 加载配置
config_path = Path.cwd() / ".." / "config.json"
with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

MODEL_CONFIG = config["model"]

print("✅ 配置加载成功")

✅ 配置加载成功


---

## 🔧 2. 定义工具

In [2]:
# 工具描述
tools = [
    {
        "name": "get_weather",
        "description": "获取指定城市的天气信息",
        "parameters": {
            "city": {
                "type": "string",
                "description": "城市名称，如北京、上海"
            }
        }
    },
    {
        "name": "calculate",
        "description": "进行数学计算",
        "parameters": {
            "expression": {
                "type": "string",
                "description": "数学表达式，如 '2 + 3 * 4'"
            }
        }
    },
    {
        "name": "get_time",
        "description": "获取当前时间",
        "parameters": {}
    }
]

# 工具实现
def get_weather(city: str) -> str:
    """获取天气"""
    weather_database = {
        "北京": {"天气": "晴天", "温度": "25°C"},
        "上海": {"天气": "多云", "温度": "23°C"},
        "广州": {"天气": "小雨", "温度": "28°C"}
    }
    if city in weather_database:
        data = weather_database[city]
        return f"{city}的天气：{data['天气']}，温度{data['温度']}"
    return f"未知城市: {city}"

def calculate(expression: str) -> str:
    """计算器"""
    try:
        allowed_chars = "0123456789+-*/(). "
        if all(c in allowed_chars for c in expression):
            return f"{expression} = {eval(expression)}"
        return "非法表达式"
    except:
        return "计算错误"

def get_time() -> str:
    """获取时间"""
    from datetime import datetime
    return datetime.now().strftime("%Y年%m月%d日 %H:%M:%S")

# 工具映射
tool_map = {
    "get_weather": get_weather,
    "calculate": calculate,
    "get_time": get_time
}

print("✅ 工具定义完成")
print(f"   可用工具: {list(tool_map.keys())}")

✅ 工具定义完成
   可用工具: ['get_weather', 'calculate', 'get_time']


---

## 🚀 3. 定义 LLM 调用和解析函数

In [3]:
def call_llm(messages: list) -> str:
    """调用 LLM"""
    url = f"{MODEL_CONFIG['base_url']}/v1/chat/completions"
    payload = {
        "model": MODEL_CONFIG['name'],
        "messages": messages,
        "temperature": 0.0,
        "max_tokens": 200
    }
    try:
        response = requests.post(url, json=payload, timeout=60)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"]
    except Exception as e:
        return f"❌ 错误：{str(e)}"

def extract_json(response: str) -> dict:
    """提取 JSON"""
    try:
        return json.loads(response)
    except:
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except:
                return None
        return None

print("✅ LLM 调用和解析函数定义完成")

✅ LLM 调用和解析函数定义完成


---

## 🎯 4. 单步 Agent 核心逻辑

In [4]:
def run_single_step_agent(user_input: str) -> str:
    """
    运行单步 Agent
    """
    # 构建工具描述
    tools_description = "\n".join([
        f"{i+1}. {tool['name']}: {tool['description']}"
        for i, tool in enumerate(tools)
    ])
    
    system_prompt = f"""
你是一个智能助手，可以使用工具来帮助回答问题。

可用工具：
{tools_description}

请输出结构化的 JSON 响应：
{{
  "thought": "你的思考过程",
  "action": "工具名称或 'finish'",
  "params": {{参数对象}}
}}

如果不需要工具，action 填 "finish"，params 为空对象。
"""
    
    # 第一步：获取模型的决策
    messages = [
        {"role": "system", "content": system_prompt.strip()},
        {"role": "user", "content": user_input}
    ]
    
    response = call_llm(messages)
    action = extract_json(response)
    
    if not action:
        return f"模型响应无法解析: {response}"
    
    print(f"🤔 思考: {action.get('thought', '')}")
    print(f"⚡ 行动: {action.get('action', '')}")
    
    # 第二步：执行工具或直接回答
    if action["action"] == "finish":
        # 不需要调用工具，直接生成回答
        return call_llm([{"role": "user", "content": user_input}])
    else:
        # 调用工具
        tool_name = action["action"]
        params = action.get("params", {})
        
        if tool_name not in tool_map:
            return f"未知工具: {tool_name}"
        
        # 调用工具
        tool_func = tool_map[tool_name]
        try:
            tool_result = tool_func(**params)
            print(f"🔧 工具结果: {tool_result}")
            
            # 第三步：用工具结果生成最终回答
            summary_prompt = f"根据以下工具执行结果，用自然语言回答用户问题：\n\n工具结果：{tool_result}\n\n用户问题：{user_input}"
            return call_llm([{"role": "user", "content": summary_prompt}])
        
        except Exception as e:
            return f"工具调用失败: {str(e)}"

print("✅ 单步 Agent 定义完成")

✅ 单步 Agent 定义完成


---

## 🎮 5. 测试单步 Agent

In [5]:
# 测试 1：天气查询
print("\n=== 测试 1：天气查询 ===")
answer = run_single_step_agent("北京今天天气怎么样？")
print(f"🤖 回答：{answer}")


=== 测试 1：天气查询 ===
🤔 思考: 用户询问北京今天的天气情况，我需要使用 get_weather 工具来获取北京的天气信息。
⚡ 行动: get_weather
🔧 工具结果: 北京的天气：晴天，温度25°C
🤖 回答：Thinking Process:

1.  **Analyze the Request:**
    *   Input: Tool execution result (北京的天气：晴天，温度 25°C) and User question (北京今天天气怎么样？).
    *   Task: Answer the user's question in natural language based on the tool result.
    *   Language: Chinese.

2.  **Analyze the Tool Result:**
    *   Location: 北京 (Beijing)
    *   Weather Condition: 晴天 (Sunny/Clear)
    *   Temperature: 25°C

3.  **Analyze the User Question:**
    *   Question: 北京今天天气怎么样？ (How is the weather in Beijing today?)
    *   Intent: Seeking weather information for Beijing.

4.  **Synthesize the Answer:**
    *   Combine the information from the tool result into a natural sentence.
    *   Draft 1: 北京


In [6]:
# 测试 2：数学计算
print("\n=== 测试 2：数学计算 ===")
answer = run_single_step_agent("123 + 456 等于多少？")
print(f"🤖 回答：{answer}")


=== 测试 2：数学计算 ===
🤔 思考: 用户问的是一个简单的数学加法问题，我需要使用 calculate 工具来计算 123 + 456 的结果。
⚡ 行动: calculate
🔧 工具结果: 123 + 456 = 579
🤖 回答：Thinking Process:

1.  **Analyze the Request:**
    *   Input: Tool execution result ("123 + 456 = 579") and User question ("123 + 456 等于多少？" which means "What is 123 + 456 equal to?").
    *   Task: Answer the user's question in natural language based on the tool result.
    *   Language: Chinese (matching the user's question).

2.  **Analyze the Tool Result:**
    *   Content: "123 + 456 = 579"
    *   Meaning: The sum of 123 and 456 is 579.

3.  **Formulate the Answer:**
    *   Direct answer: 123 + 456 equals 579.
    *   Natural language


In [7]:
# 测试 3：不需要工具
print("\n=== 测试 3：不需要工具 ===")
answer = run_single_step_agent("你好，介绍一下你自己")
print(f"🤖 回答：{answer}")


=== 测试 3：不需要工具 ===
🤔 思考: 用户要求我介绍自己，这是一个简单的自我介绍请求，不需要使用任何工具。我可以直接回答。
⚡ 行动: finish
🤖 回答：嗯，用户让我介绍一下自己。首先，我需要确认用户的需求是什么。可能他们刚接触这个模型，想了解我的能力和特点。我应该简明扼要地介绍自己的身份、主要功能和优势。

我是 Qwen3.5，是通义千问系列的最新版本。需要突出我的升级点，比如上下文窗口、多语言支持、推理能力等。但用户可能不需要太技术化的细节，所以得用通俗易懂的语言。

要分点说明，但用户可能希望结构清晰。可能需要提到我的核心能力，比如对话理解、代码生成、多模态处理等。同时，要强调应用场景，比如工作、学习、生活等，让用户知道我能帮他们做什么。

还要注意语气友好，避免过于机械。可能需要加入一些例子，比如处理文档、写代码、分析图表等。但用户的问题比较直接，可能不需要太长的回答，保持简洁。

需要检查是否有遗漏的重要信息，比如知识截止时间、多语言支持数量、上下文长度


---

## ✅ 本阶段总结

你已经完成了：

1. ✅ 实现了单步 Agent
2. ✅ 模型可以自动决定是否调用工具
3. ✅ 学会了工具描述的格式

---

## 🔗 下一步

进入 **[第 5 阶段：Agent Loop](../step5/step5.ipynb)**，实现完整的思考-行动-观察循环。